# ACL Tear Detection — V8.4 MRNet (Simplified + Anti-Overfit)

**V8.3 problems:** 10-epoch majority-class collapse + explosive overfitting (22% gap)

**V8.4 fixes:**
| Setting | V8.3 | V8.4 |
|---------|------|------|
| Loss | CrossEntropy (class-weighted) | **Focal Loss (α=0.75, γ=2) + label smoothing** |
| Grad accumulation | 1 | **8 (effective batch=8)** |
| Backbone LR | 2e-5 | **1e-5** |
| Classifier LR | 1e-4 | **5e-5** |
| Weight decay | 0.01 | **0.05** |
| Classifier | Linear(1280,2) | **Bottleneck: 1280→256→BN→ReLU→Drop→2** |
| Pooling | Max | **Attention-weighted** |
| Warmup | 3 epochs | **5 epochs** |
| Priyank + K-fold | Yes | **Removed (simplified)** |

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# ============================================================
# Configuration
# ============================================================
MRNET_DATA_DIR = '/content/drive/MyDrive/dataset/mrnet_sagittal'
BATCH_SIZE = 1
NUM_EPOCHS = 50
RANDOM_SEED = 42
MAX_SLICES = 40

# V8.4 Settings
BACKBONE_LR = 1e-5
CLASSIFIER_LR = 5e-5
WEIGHT_DECAY = 0.05
DROPOUT = 0.5
FREEZE_BLOCKS = 3
WARMUP_EPOCHS = 5
PATIENCE = 10
ACCUMULATION_STEPS = 8
LABEL_SMOOTHING = 0.1
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR, SequentialLR
import torchvision.transforms as transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# ============================================================
# Load Metadata
# ============================================================
mrnet_path = Path(MRNET_DATA_DIR)
mrnet_df = pd.read_csv(mrnet_path / 'metadata.csv')
mrnet_df['label_binary'] = mrnet_df['label'].astype(int)
mrnet_df['label_name_binary'] = mrnet_df['label_binary'].map({0: 'Normal', 1: 'Tear'})

print(f"MRNet Dataset: {len(mrnet_df)} patients")
print(mrnet_df['label_name_binary'].value_counts())
print(f"\nSlice count: min={mrnet_df['num_slices'].min()}, max={mrnet_df['num_slices'].max()}, mean={mrnet_df['num_slices'].mean():.1f}")


In [ ]:
# ============================================================
# Dataset
# ============================================================
class MRNetVolumeDataset(Dataset):
    def __init__(self, df, data_dir, max_slices=MAX_SLICES, augment=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.max_slices = max_slices
        self.augment = augment
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )
        self.valid_indices = []
        for idx, row in self.df.iterrows():
            if (self.data_dir / row['filename']).exists():
                self.valid_indices.append(idx)
        print(f"  {len(self.valid_indices)} valid patients")

    def __len__(self):
        return len(self.valid_indices)

    def _augment_volume(self, slices):
        import cv2
        if np.random.random() < 0.5:
            slices = slices[:, :, ::-1].copy()
        if np.random.random() < 0.5:
            angle = np.random.uniform(-15, 15)
            h, w = slices.shape[1], slices.shape[2]
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
            slices = np.stack([cv2.warpAffine(s, M, (w, h), borderValue=0) for s in slices])
        if np.random.random() < 0.4:
            slices = np.clip(slices + np.random.uniform(-0.1, 0.1), 0, 1)
        if np.random.random() < 0.4:
            contrast = np.random.uniform(0.8, 1.2)
            mean_val = slices.mean()
            slices = np.clip((slices - mean_val) * contrast + mean_val, 0, 1)
        if np.random.random() < 0.3:
            slices = np.clip(slices + np.random.normal(0, 0.02, slices.shape).astype(np.float32), 0, 1)
        return slices

    def __getitem__(self, idx):
        patient_idx = self.valid_indices[idx]
        row = self.df.iloc[patient_idx]
        volume = np.load(self.data_dir / row['filename'])['data']
        slices = volume.astype(np.float32) / 255.0
        actual_slices = slices.shape[0]

        if actual_slices < self.max_slices:
            pad = np.zeros((self.max_slices - actual_slices, *slices.shape[1:]), dtype=np.float32)
            slices = np.concatenate([slices, pad], axis=0)
        elif actual_slices > self.max_slices:
            offset = (actual_slices - self.max_slices) // 2
            slices = slices[offset:offset + self.max_slices]
            actual_slices = self.max_slices

        if self.augment:
            slices = self._augment_volume(slices)

        slices_3ch = np.stack((slices,)*3, axis=1)
        slices_tensor = torch.FloatTensor(slices_3ch)
        slices_tensor = torch.stack([self.normalize(s) for s in slices_tensor])
        label = int(row['label_binary'])
        return slices_tensor, label, patient_idx, actual_slices

    def get_labels(self):
        return [int(self.df.iloc[i]['label_binary']) for i in self.valid_indices]


In [ ]:
# ============================================================
# Focal Loss
# ============================================================
class FocalLoss(nn.Module):
    """Down-weights easy examples, focuses on hard/minority examples."""
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        # Convert to binary (class 1 = Tear)
        probs = torch.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, num_classes=2).float()

        # Label smoothing
        if self.label_smoothing > 0:
            targets_onehot = targets_onehot * (1 - self.label_smoothing) + self.label_smoothing / 2

        # Per-class alpha weights: [1-alpha for Normal, alpha for Tear]
        alpha_weight = torch.tensor([1 - self.alpha, self.alpha], device=logits.device)
        alpha_t = (targets_onehot * alpha_weight).sum(dim=1)

        # Focal modulation
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = (targets_onehot * probs).sum(dim=1)  # probability of true class
        focal_weight = alpha_t * (1 - pt) ** self.gamma

        return (focal_weight * ce).mean()


In [ ]:
# ============================================================
# Attention Pooling
# ============================================================
class AttentionPool(nn.Module):
    """Learns which slices are important instead of raw max-pool."""
    def __init__(self, dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

    def forward(self, x, mask=None):
        # x: (B, S, D)
        attn = self.attention(x)  # (B, S, 1)
        if mask is not None:
            attn = attn.masked_fill(mask.unsqueeze(2), float('-inf'))
        attn = torch.softmax(attn, dim=1)
        return (attn * x).sum(dim=1)  # (B, D)


In [ ]:
# ============================================================
# Model — V8.4: Bottleneck classifier + Attention pooling
# ============================================================
class ACLVolumeNet(nn.Module):
    def __init__(self, dropout=DROPOUT, freeze_blocks=FREEZE_BLOCKS):
        super().__init__()
        backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.features = backbone.features
        self.pool = backbone.avgpool
        self.in_features = backbone.classifier[1].in_features  # 1280

        # Freeze first N blocks
        for i in range(min(freeze_blocks, len(self.features))):
            for param in self.features[i].parameters():
                param.requires_grad = False

        # Attention pooling over slices
        self.attn_pool = AttentionPool(self.in_features)

        # Bottleneck classifier
        self.classifier = nn.Sequential(
            nn.Linear(self.in_features, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(256, 2)
        )

        frozen = sum(p.numel() for p in self.parameters() if not p.requires_grad)
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = frozen + trainable
        print(f"  Backbone: EfficientNet-B0, blocks 0-{freeze_blocks-1} FROZEN")
        print(f"  Trainable blocks: {freeze_blocks}-{len(self.features)-1} + attn_pool + classifier")
        print(f"  Classifier: Linear(1280,256)->LN->ReLU->Drop({dropout})->Linear(256,2)")
        print(f"  Pooling: Attention-weighted (learned)")
        print(f"  Params: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)")

    def forward(self, x, actual_slices=None):
        B, S, C, H, W = x.shape
        x = x.view(B * S, C, H, W)
        features = self.features(x)
        features = self.pool(features)
        features = features.flatten(1)
        features = features.view(B, S, -1)

        # Build mask for padded slices
        mask = None
        if actual_slices is not None:
            mask = torch.arange(S, device=features.device).unsqueeze(0) >= actual_slices.unsqueeze(1)

        # Attention-weighted pooling
        pooled = self.attn_pool(features, mask=mask)
        logits = self.classifier(pooled)
        return logits

    def get_param_groups(self, backbone_lr, classifier_lr):
        backbone_params = []
        head_params = []
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            if 'classifier' in name or 'attn_pool' in name:
                head_params.append(param)
            else:
                backbone_params.append(param)
        return [
            {'params': backbone_params, 'lr': backbone_lr},
            {'params': head_params, 'lr': classifier_lr},
        ]


In [ ]:
# ============================================================
# Scheduler
# ============================================================
def create_scheduler(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = total_epochs * steps_per_epoch
    min_factor = 0.1

    def warmup_fn(step):
        if step < warmup_steps:
            return min_factor + (1.0 - min_factor) * float(step) / float(max(1, warmup_steps))
        return 1.0

    warmup_scheduler = LambdaLR(optimizer, lr_lambda=warmup_fn)
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=1e-7)
    return SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])


In [ ]:
# ============================================================
# Training with Gradient Accumulation
# ============================================================
def train_epoch(model, loader, criterion, optimizer, scheduler, device, accum_steps=ACCUMULATION_STEPS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []

    optimizer.zero_grad()
    for i, (volumes, labels, _, actual_slices) in enumerate(tqdm(loader, desc='Training', leave=False)):
        volumes = volumes.to(device)
        labels = labels.to(device)
        actual_slices = actual_slices.to(device)

        logits = model(volumes, actual_slices=actual_slices)
        loss = criterion(logits, labels) / accum_steps
        loss.backward()

        if (i + 1) % accum_steps == 0 or (i + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * accum_steps * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        probs = torch.softmax(logits, dim=1)[:, 1]
        all_probs.extend(probs.detach().cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / total
    acc = 100.0 * correct / total
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    return avg_loss, acc, auc


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for volumes, labels, _, actual_slices in tqdm(loader, desc='Validating', leave=False):
            volumes = volumes.to(device)
            labels = labels.to(device)
            actual_slices = actual_slices.to(device)

            logits = model(volumes, actual_slices=actual_slices)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / total
    acc = 100.0 * correct / total
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    return avg_loss, acc, auc, all_preds, all_labels


In [ ]:
# ============================================================
# Train/Val/Test Split (70/15/15)
# ============================================================
train_df, temp_df = train_test_split(
    mrnet_df, test_size=0.3, stratify=mrnet_df['label_binary'], random_state=RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label_binary'], random_state=RANDOM_SEED
)

print("MRNet Dataset Split:")
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    dist = df['label_name_binary'].value_counts().to_dict()
    print(f"  {name}: {len(df)} patients — {dist}")


In [ ]:
# ============================================================
# Create Datasets & DataLoaders
# ============================================================
print("Creating datasets...")
print(f"  MAX_SLICES={MAX_SLICES}")

print("\nTrain (augmented):")
train_dataset = MRNetVolumeDataset(train_df, MRNET_DATA_DIR, augment=True)
print("Val:")
val_dataset = MRNetVolumeDataset(val_df, MRNET_DATA_DIR, augment=False)
print("Test:")
test_dataset = MRNetVolumeDataset(test_df, MRNET_DATA_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain: {len(train_dataset)} patients, {len(train_loader)} batches")
print(f"Val:   {len(val_dataset)} patients")
print(f"Test:  {len(test_dataset)} patients")


In [ ]:
# ============================================================
# Model, Loss, Optimizer
# ============================================================
print("Creating V8.4 model...")
model = ACLVolumeNet()
model = model.to(device)

# Focal Loss with label smoothing
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)

# Differential learning rates
param_groups = model.get_param_groups(BACKBONE_LR, CLASSIFIER_LR)
optimizer = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

# Warmup + Cosine Annealing (steps adjusted for accumulation)
effective_steps_per_epoch = len(train_loader) // ACCUMULATION_STEPS
scheduler = create_scheduler(optimizer, warmup_epochs=WARMUP_EPOCHS, total_epochs=NUM_EPOCHS, steps_per_epoch=effective_steps_per_epoch)

print(f"\nV8.4 Settings:")
print(f"  Loss:          Focal Loss (alpha={FOCAL_ALPHA}, gamma={FOCAL_GAMMA}) + label_smoothing={LABEL_SMOOTHING}")
print(f"  Optimizer:     AdamW (bb_lr={BACKBONE_LR}, cl_lr={CLASSIFIER_LR}, wd={WEIGHT_DECAY})")
print(f"  Scheduler:     Warmup ({WARMUP_EPOCHS} epochs) + Cosine Annealing")
print(f"  Grad Accum:    {ACCUMULATION_STEPS} steps (effective batch={ACCUMULATION_STEPS})")
print(f"  Backbone:      Blocks 0-{FREEZE_BLOCKS-1} frozen")
print(f"  Pooling:       Attention-weighted")
print(f"  Classifier:    Bottleneck (1280->256->2)")
print(f"  Early stop:    patience={PATIENCE} on val_AUC")


In [ ]:
# ============================================================
# Training Loop
# ============================================================
history = {'train_loss': [], 'train_acc': [], 'train_auc': [],
           'val_loss': [], 'val_acc': [], 'val_auc': [], 'lr': []}
best_val_auc = 0.0
patience_counter = 0

SAVE_PATH = '/content/drive/MyDrive/dataset/best_acl_model_v8.4.pth'

print(f"Training for up to {NUM_EPOCHS} epochs (patience={PATIENCE})...\n")

for epoch in range(NUM_EPOCHS):
    current_lr_bb = optimizer.param_groups[0]['lr']
    current_lr_cl = optimizer.param_groups[1]['lr']

    train_loss, train_acc, train_auc = train_epoch(
        model, train_loader, criterion, optimizer, scheduler, device
    )
    val_loss, val_acc, val_auc, val_preds, val_labels = validate(
        model, val_loader, criterion, device
    )

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['lr'].append(current_lr_bb)

    gap = train_acc - val_acc
    gap_warn = ' OVERFIT' if gap > 10 else ''

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  (bb_lr={current_lr_bb:.1e}, cl_lr={current_lr_cl:.1e})")
    print(f"  Train: loss={train_loss:.4f}  acc={train_acc:.2f}%  AUC={train_auc:.4f}")
    print(f"  Val:   loss={val_loss:.4f}  acc={val_acc:.2f}%  AUC={val_auc:.4f}  (gap={gap:.1f}%){gap_warn}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  -> Saved best (AUC={val_auc:.4f}, acc={val_acc:.2f}%, loss={val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print(f"\nBest val AUC: {best_val_auc:.4f}")


In [ ]:
# ============================================================
# Plot Training History
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(30, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(history['train_auc'], label='Train')
axes[2].plot(history['val_auc'], label='Val')
axes[2].set_title('AUC'); axes[2].legend(); axes[2].grid(True)

gaps = [t - v for t, v in zip(history['train_acc'], history['val_acc'])]
axes[3].plot(gaps, color='red')
axes[3].axhline(y=10, color='orange', linestyle='--', label='10% threshold')
axes[3].set_title('Overfit Gap'); axes[3].legend(); axes[3].grid(True)

axes[4].plot(history['lr'], color='purple')
axes[4].set_title('Learning Rate (backbone)'); axes[4].grid(True)
axes[4].set_yscale('log')

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dataset/training_history_v8.4.png', dpi=150)
plt.show()


In [ ]:
# ============================================================
# Evaluate on Test Set
# ============================================================
model.load_state_dict(torch.load(SAVE_PATH, weights_only=True))
test_loss, test_acc, test_auc, test_preds, test_labels = validate(
    model, test_loader, criterion, device
)

label_names = ['Normal', 'Tear']

print('=' * 60)
print('TEST RESULTS — V8.4 (MRNet)')
print('=' * 60)
print(f'Accuracy: {test_acc:.2f}%')
print(f'AUC: {test_auc:.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — MRNet Test (V8.4)')
plt.savefig('/content/drive/MyDrive/dataset/confusion_matrix_v8.4.png', dpi=150)
plt.show()
